# Experiment 5: Hadoop Installation, Configuration, and HDFS Operations

**Aim:** To install and configure Hadoop in a single-node environment and perform basic HDFS operations using command-line tools.

**Mapped CO(s):** CO2

## 1. Import Required Libraries

In [1]:
import os
import shutil
import subprocess
import time
from pathlib import Path
from shlex import quote

## 2. Hadoop Ecosystem (Brief Theory)

Apache Hadoop is an open-source framework for storing and processing large datasets in a distributed, fault-tolerant way.

| Component | Role |
|---|---|
| **HDFS** | Hadoop Distributed File System — stores data as blocks across nodes |
| **YARN** | Resource manager and job scheduler |
| **MapReduce** | Parallel processing model |
| **Hadoop Common** | Shared utilities and libraries |

This experiment uses a **single-node (pseudo-distributed)** setup: NameNode and DataNode run on the same machine, with replication factor **1**.

Hadoop 3.4.1 is installed at `/Users/ojas/hadoop-3.4.1` (a path without spaces). Java 17 is used from Homebrew OpenJDK.

All files for this experiment stay inside the `Experiment5` folder: input files in `local_files/`, downloaded files in `downloads/`, and Hadoop configuration, logs and HDFS block storage in `hadoop_runtime/`.

## 3. Set Java and Hadoop Environment

In [2]:
EXP_DIR = Path("/Users/ojas/Desktop/DS Lab/Experiment5")

# Hadoop's scripts and shell break on paths containing spaces ("DS Lab"),
# so every path Hadoop sees goes through this space-free symlink to EXP_DIR.
EXP_LINK = Path("/tmp/exp5")
if EXP_LINK.is_symlink() or EXP_LINK.exists():
    EXP_LINK.unlink()
EXP_LINK.symlink_to(EXP_DIR)

LOCAL_DIR = EXP_LINK / "local_files"
DOWNLOAD_DIR = EXP_LINK / "downloads"
RUNTIME_DIR = EXP_LINK / "hadoop_runtime"
HDFS_DATA = RUNTIME_DIR / "hdfs"
CONF_DIR = RUNTIME_DIR / "conf"
LOG_DIR = RUNTIME_DIR / "logs"
PID_DIR = RUNTIME_DIR / "pids"
TMP_DIR = RUNTIME_DIR / "tmp"

JAVA_HOME = Path("/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home")
HADOOP_HOME = Path("/Users/ojas/hadoop-3.4.1")

os.environ["JAVA_HOME"] = str(JAVA_HOME)
os.environ["HADOOP_HOME"] = str(HADOOP_HOME)
os.environ["HADOOP_CONF_DIR"] = str(CONF_DIR)
os.environ["HADOOP_LOG_DIR"] = str(LOG_DIR)
os.environ["HADOOP_PID_DIR"] = str(PID_DIR)
os.environ["PATH"] = (
    f"{JAVA_HOME / 'bin'}:{HADOOP_HOME / 'bin'}:{HADOOP_HOME / 'sbin'}:"
    + os.environ["PATH"]
)

for folder in (LOG_DIR, PID_DIR, TMP_DIR, HDFS_DATA, DOWNLOAD_DIR):
    folder.mkdir(parents=True, exist_ok=True)

if not (CONF_DIR / "log4j.properties").exists():
    if CONF_DIR.exists():
        shutil.rmtree(CONF_DIR)
    shutil.copytree(HADOOP_HOME / "etc" / "hadoop", CONF_DIR)


def run(cmd, check=True, quiet=False):
    if not quiet:
        print(f"$ {cmd}")
    result = subprocess.run(
        cmd,
        shell=True,
        text=True,
        capture_output=True,
        env=os.environ.copy(),
    )
    out = (result.stdout or "").rstrip()
    err = (result.stderr or "").rstrip()
    if out:
        print(out)
    if err:
        err = "\n".join(
            line for line in err.splitlines()
            if "NativeCodeLoader" not in line
        )
        if err:
            print(err)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {cmd}")
    return result

print("JAVA_HOME      :", os.environ["JAVA_HOME"])
print("HADOOP_HOME    :", os.environ["HADOOP_HOME"])
print("HADOOP_CONF_DIR:", os.environ["HADOOP_CONF_DIR"])
print("Runtime files  :", EXP_DIR / "hadoop_runtime")

JAVA_HOME      : /opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home
HADOOP_HOME    : /Users/ojas/hadoop-3.4.1
HADOOP_CONF_DIR: /tmp/exp5/hadoop_runtime/conf
Runtime files  : /Users/ojas/Desktop/DS Lab/Experiment5/hadoop_runtime


## 4. Check Java and Hadoop Versions

In [3]:
run("java -version")
print()
run("hadoop version")

$ java -version
openjdk version "17.0.19" 2026-04-21
OpenJDK Runtime Environment Homebrew (build 17.0.19+0)
OpenJDK 64-Bit Server VM Homebrew (build 17.0.19+0, mixed mode, sharing)

$ hadoop version
Hadoop 3.4.1
Source code repository https://github.com/apache/hadoop.git -r 4d7825309348956336b8f06a08322b78422849b1
Compiled by mthakur on 2024-10-09T14:57Z
Compiled on platform linux-x86_64
Compiled with protoc 3.23.4
From source with checksum 7292fe9dba5e2e44e3a9f763fce3e680
This command was run using /Users/ojas/hadoop-3.4.1/share/hadoop/common/hadoop-common-3.4.1.jar


CompletedProcess(args='hadoop version', returncode=0, stdout='Hadoop 3.4.1\nSource code repository https://github.com/apache/hadoop.git -r 4d7825309348956336b8f06a08322b78422849b1\nCompiled by mthakur on 2024-10-09T14:57Z\nCompiled on platform linux-x86_64\nCompiled with protoc 3.23.4\nFrom source with checksum 7292fe9dba5e2e44e3a9f763fce3e680\nThis command was run using /Users/ojas/hadoop-3.4.1/share/hadoop/common/hadoop-common-3.4.1.jar\n', stderr='')

## 5. Configure Hadoop XML Files

Default Hadoop configuration is copied, then these properties are set:

- `core-site.xml`: `fs.defaultFS = hdfs://localhost:9000`
- `hdfs-site.xml`: replication = 1, NameNode and DataNode directories

In [4]:
default_conf = HADOOP_HOME / "etc" / "hadoop"
if CONF_DIR.exists():
    shutil.rmtree(CONF_DIR)
shutil.copytree(default_conf, CONF_DIR)

env_sh = CONF_DIR / "hadoop-env.sh"
env_text = env_sh.read_text()
if "EXP5 JAVA_HOME" not in env_text:
    env_sh.write_text(
        env_text + f"\n# EXP5 JAVA_HOME\nexport JAVA_HOME={JAVA_HOME}\n"
    )

(CONF_DIR / "core-site.xml").write_text(f"""<?xml version="1.0" encoding="UTF-8"?>
<?xml-stylesheet type="text/xsl" href="configuration.xsl"?>
<configuration>
  <property>
    <name>fs.defaultFS</name>
    <value>hdfs://localhost:9000</value>
  </property>
  <property>
    <name>hadoop.tmp.dir</name>
    <value>{TMP_DIR}</value>
  </property>
</configuration>
""")

(CONF_DIR / "hdfs-site.xml").write_text(f"""<?xml version="1.0" encoding="UTF-8"?>
<?xml-stylesheet type="text/xsl" href="configuration.xsl"?>
<configuration>
  <property>
    <name>dfs.replication</name>
    <value>1</value>
  </property>
  <property>
    <name>dfs.namenode.name.dir</name>
    <value>file://{HDFS_DATA / 'namenode'}</value>
  </property>
  <property>
    <name>dfs.datanode.data.dir</name>
    <value>file://{HDFS_DATA / 'datanode'}</value>
  </property>
  <property>
    <name>dfs.namenode.http-address</name>
    <value>localhost:9870</value>
  </property>
  <property>
    <name>dfs.datanode.address</name>
    <value>localhost:9866</value>
  </property>
  <property>
    <name>dfs.datanode.http.address</name>
    <value>localhost:9864</value>
  </property>
  <property>
    <name>dfs.datanode.ipc.address</name>
    <value>localhost:9867</value>
  </property>
  <property>
    <name>dfs.permissions.enabled</name>
    <value>false</value>
  </property>
</configuration>
""")

print("Configuration written to", CONF_DIR)
print("\n--- core-site.xml ---")
print((CONF_DIR / "core-site.xml").read_text())
print("--- hdfs-site.xml ---")
print((CONF_DIR / "hdfs-site.xml").read_text())

Configuration written to /tmp/exp5/hadoop_runtime/conf

--- core-site.xml ---
<?xml version="1.0" encoding="UTF-8"?>
<?xml-stylesheet type="text/xsl" href="configuration.xsl"?>
<configuration>
  <property>
    <name>fs.defaultFS</name>
    <value>hdfs://localhost:9000</value>
  </property>
  <property>
    <name>hadoop.tmp.dir</name>
    <value>/tmp/exp5/hadoop_runtime/tmp</value>
  </property>
</configuration>

--- hdfs-site.xml ---
<?xml version="1.0" encoding="UTF-8"?>
<?xml-stylesheet type="text/xsl" href="configuration.xsl"?>
<configuration>
  <property>
    <name>dfs.replication</name>
    <value>1</value>
  </property>
  <property>
    <name>dfs.namenode.name.dir</name>
    <value>file:///tmp/exp5/hadoop_runtime/hdfs/namenode</value>
  </property>
  <property>
    <name>dfs.datanode.data.dir</name>
    <value>file:///tmp/exp5/hadoop_runtime/hdfs/datanode</value>
  </property>
  <property>
    <name>dfs.namenode.http-address</name>
    <value>localhost:9870</value>
  </property>


## 6. Format the Hadoop NameNode

In [5]:
run("hdfs --daemon stop datanode", check=False, quiet=True)
run("hdfs --daemon stop namenode", check=False, quiet=True)
time.sleep(2)

if HDFS_DATA.exists():
    shutil.rmtree(HDFS_DATA)
HDFS_DATA.mkdir(parents=True, exist_ok=True)

run("hdfs namenode -format -force -nonInteractive")

$ hdfs namenode -format -force -nonInteractive
2026-10-23 14:25:10 INFO  NameNode:809 - STARTUP_MSG: 
/************************************************************
STARTUP_MSG: Starting NameNode
STARTUP_MSG:   host = Ojass-MacBook-Air.local/127.0.0.1
STARTUP_MSG:   args = [-format, -force, -nonInteractive]
STARTUP_MSG:   version = 3.4.1
STARTUP_MSG:   classpath = /tmp/exp5/hadoop_runtime/conf:/Users/ojas/hadoop-3.4.1/share/hadoop/common/lib/jaxb-impl-2.2.3-1.jar:/Users/ojas/hadoop-3.4.1/share/hadoop/common/lib/jetty-webapp-9.4.53.v20231009.jar:/Users/ojas/hadoop-3.4.1/share/hadoop/common/lib/curator-client-5.2.0.jar:/Users/ojas/hadoop-3.4.1/share/hadoop/common/lib/netty-handler-4.1.100.Final.jar:/Users/ojas/hadoop-3.4.1/share/hadoop/common/lib/kerby-pkix-2.0.3.jar:/Users/ojas/hadoop-3.4.1/share/hadoop/common/lib/commons-logging-1.2.jar:/Users/ojas/hadoop-3.4.1/share/hadoop/common/lib/reload4j-1.2.22.jar:/Users/ojas/hadoop-3.4.1/share/hadoop/common/lib/jackson-core-2.12.7.jar:/Users/oja

CompletedProcess(args='hdfs namenode -format -force -nonInteractive', returncode=0, stdout="2026-10-23 14:25:10 INFO  NameNode:809 - STARTUP_MSG: \n/************************************************************\nSTARTUP_MSG: Starting NameNode\nSTARTUP_MSG:   host = Ojass-MacBook-Air.local/127.0.0.1\nSTARTUP_MSG:   args = [-format, -force, -nonInteractive]\nSTARTUP_MSG:   version = 3.4.1\nSTARTUP_MSG:   classpath = /tmp/exp5/hadoop_runtime/conf:/Users/ojas/hadoop-3.4.1/share/hadoop/common/lib/jaxb-impl-2.2.3-1.jar:/Users/ojas/hadoop-3.4.1/share/hadoop/common/lib/jetty-webapp-9.4.53.v20231009.jar:/Users/ojas/hadoop-3.4.1/share/hadoop/common/lib/curator-client-5.2.0.jar:/Users/ojas/hadoop-3.4.1/share/hadoop/common/lib/netty-handler-4.1.100.Final.jar:/Users/ojas/hadoop-3.4.1/share/hadoop/common/lib/kerby-pkix-2.0.3.jar:/Users/ojas/hadoop-3.4.1/share/hadoop/common/lib/commons-logging-1.2.jar:/Users/ojas/hadoop-3.4.1/share/hadoop/common/lib/reload4j-1.2.22.jar:/Users/ojas/hadoop-3.4.1/share/h

## 7. Start Hadoop HDFS Services

In [6]:
run("hdfs --daemon start namenode")
run("hdfs --daemon start datanode")

for _ in range(20):
    time.sleep(1)
    report = subprocess.run(
        "hdfs dfsadmin -report",
        shell=True,
        text=True,
        capture_output=True,
        env=os.environ.copy(),
    )
    if report.returncode == 0 and "Live datanodes" in (report.stdout + report.stderr):
        break

print("HDFS daemons started.\n")
run("jps")

$ hdfs --daemon start namenode
ERROR: Cannot set priority of namenode process 3385


RuntimeError: Command failed with exit code 1: hdfs --daemon start namenode

## 8. Verify HDFS Is Running

In [ ]:
run("hdfs dfsadmin -report")

$ hdfs dfsadmin -report


2026-10-09 13:24:45 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Configured Capacity: 494384795648 (460.43 GB)
Present Capacity: 93862023168 (87.42 GB)
DFS Remaining: 93862019072 (87.42 GB)
DFS Used: 4096 (4 KB)
DFS Used%: 0.00%
Replicated Blocks:
	Under replicated blocks: 0
	Blocks with corrupt replicas: 0
	Missing blocks: 0
	Missing blocks (with replication factor 1): 0
	Low redundancy blocks with highest priority to recover: 0
	Pending deletion blocks: 0
Erasure Coded Block Groups: 
	Low redundancy block groups: 0
	Block groups with corrupt internal blocks: 0
	Missing block groups: 0
	Low redundancy blocks with highest priority to recover: 0
	Pending deletion blocks: 0

-------------------------------------------------
Live datanodes (1):

Name: 127.0.0.1:9866 (localhost)
Hostname: localhost
Decommission Status : Normal
Configured Capacity: 494384795648 (460.43 GB)
DFS Used: 4096 (4 KB)
Non DFS Used

CompletedProcess(args='hdfs dfsadmin -report', returncode=0, stdout='2026-10-09 13:24:45 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable\nConfigured Capacity: 494384795648 (460.43 GB)\nPresent Capacity: 93862023168 (87.42 GB)\nDFS Remaining: 93862019072 (87.42 GB)\nDFS Used: 4096 (4 KB)\nDFS Used%: 0.00%\nReplicated Blocks:\n\tUnder replicated blocks: 0\n\tBlocks with corrupt replicas: 0\n\tMissing blocks: 0\n\tMissing blocks (with replication factor 1): 0\n\tLow redundancy blocks with highest priority to recover: 0\n\tPending deletion blocks: 0\nErasure Coded Block Groups: \n\tLow redundancy block groups: 0\n\tBlock groups with corrupt internal blocks: 0\n\tMissing block groups: 0\n\tLow redundancy blocks with highest priority to recover: 0\n\tPending deletion blocks: 0\n\n-------------------------------------------------\nLive datanodes (1):\n\nName: 127.0.0.1:9866 (localhost)\nHostname: localhost\nDeco

## 9. Create Directories in HDFS

In [ ]:
run("hdfs dfs -mkdir -p /user/lab/input")
run("hdfs dfs -mkdir -p /user/lab/output")
run("hdfs dfs -ls -R /user")

$ hdfs dfs -mkdir -p /user/lab/input


2026-10-09 13:24:45 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
$ hdfs dfs -mkdir -p /user/lab/output


2026-10-09 13:24:46 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
$ hdfs dfs -ls -R /user


2026-10-09 13:24:46 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
drwxr-xr-x   - ojas supergroup          0 2026-10-09 13:24 /user/lab
drwxr-xr-x   - ojas supergroup          0 2026-10-09 13:24 /user/lab/input
drwxr-xr-x   - ojas supergroup          0 2026-10-09 13:24 /user/lab/output


CompletedProcess(args='hdfs dfs -ls -R /user', returncode=0, stdout='2026-10-09 13:24:46 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable\ndrwxr-xr-x   - ojas supergroup          0 2026-10-09 13:24 /user/lab\ndrwxr-xr-x   - ojas supergroup          0 2026-10-09 13:24 /user/lab/input\ndrwxr-xr-x   - ojas supergroup          0 2026-10-09 13:24 /user/lab/output\n', stderr='')

## 10. Upload Files to HDFS

In [ ]:
run(f"hdfs dfs -put -f {quote(str(LOCAL_DIR / 'sample.txt'))} /user/lab/input/")
run("hdfs dfs -ls /user/lab/input")

$ hdfs dfs -put -f /tmp/exp5/local_files/sample.txt /user/lab/input/


2026-10-09 13:24:47 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
$ hdfs dfs -ls /user/lab/input


2026-10-09 13:24:48 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Found 1 items
-rw-r--r--   1 ojas supergroup        262 2026-10-09 13:24 /user/lab/input/sample.txt


CompletedProcess(args='hdfs dfs -ls /user/lab/input', returncode=0, stdout='2026-10-09 13:24:48 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable\nFound 1 items\n-rw-r--r--   1 ojas supergroup        262 2026-10-09 13:24 /user/lab/input/sample.txt\n', stderr='')

## 11. Display HDFS File Contents

In [ ]:
run("hdfs dfs -cat /user/lab/input/sample.txt")

$ hdfs dfs -cat /user/lab/input/sample.txt


2026-10-09 13:24:48 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
This is a sample text file for Hadoop HDFS operations.
Experiment 5: Hadoop Installation, Configuration, and HDFS Operations

HDFS stores files by dividing them into blocks and distributing them across nodes.
Common operations: mkdir, put, ls, cat, get, cp, rm.


CompletedProcess(args='hdfs dfs -cat /user/lab/input/sample.txt', returncode=0, stdout='2026-10-09 13:24:48 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable\nThis is a sample text file for Hadoop HDFS operations.\nExperiment 5: Hadoop Installation, Configuration, and HDFS Operations\n\nHDFS stores files by dividing them into blocks and distributing them across nodes.\nCommon operations: mkdir, put, ls, cat, get, cp, rm.\n', stderr='')

## 12. Copy, Download, and Delete Files

In [ ]:
run("hdfs dfs -cp /user/lab/input/sample.txt /user/lab/output/sample_copy.txt")
print("\nCopied inside HDFS. Listing /user/lab/output:")
run("hdfs dfs -ls /user/lab/output")

print("\nDownloading to local system:")
local_copy = DOWNLOAD_DIR / "sample_from_hdfs.txt"
if local_copy.exists():
    local_copy.unlink()
run(f"hdfs dfs -get /user/lab/output/sample_copy.txt {quote(str(local_copy))}")
print("Local file contents:")
print(local_copy.read_text())

print("Deleting the copy from HDFS:")
run("hdfs dfs -rm /user/lab/output/sample_copy.txt")
run("hdfs dfs -ls /user/lab/output")

$ hdfs dfs -cp /user/lab/input/sample.txt /user/lab/output/sample_copy.txt


2026-10-09 13:24:49 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable

Copied inside HDFS. Listing /user/lab/output:
$ hdfs dfs -ls /user/lab/output


2026-10-09 13:24:49 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Found 1 items
-rw-r--r--   1 ojas supergroup        262 2026-10-09 13:24 /user/lab/output/sample_copy.txt

$ hdfs dfs -get /user/lab/output/sample_copy.txt /tmp/exp5/downloads/sample_from_hdfs.txt


2026-10-09 13:24:50 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Local file contents:
This is a sample text file for Hadoop HDFS operations.
Experiment 5: Hadoop Installation, Configuration, and HDFS Operations

HDFS stores files by dividing them into blocks and distributing them across nodes.
Common operations: mkdir, put, ls, cat, get, cp, rm.

Deleting the copy from HDFS:
$ hdfs dfs -rm /user/lab/output/sample_copy.txt


2026-10-09 13:24:51 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Deleted /user/lab/output/sample_copy.txt
$ hdfs dfs -ls /user/lab/output


2026-10-09 13:24:51 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


CompletedProcess(args='hdfs dfs -ls /user/lab/output', returncode=0, stdout='2026-10-09 13:24:51 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable\n', stderr='')

---
## Exercises / Assignments

### Exercise 1: Create a new directory in HDFS and upload multiple files

In [ ]:
run("hdfs dfs -mkdir -p /user/lab/exercise1")
run(
    "hdfs dfs -put -f "
    f"{quote(str(LOCAL_DIR / 'file1.txt'))} {quote(str(LOCAL_DIR / 'file2.txt'))} "
    "/user/lab/exercise1/"
)
run("hdfs dfs -ls /user/lab/exercise1")

$ hdfs dfs -mkdir -p /user/lab/exercise1


2026-10-09 13:24:52 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
$ hdfs dfs -put -f /tmp/exp5/local_files/file1.txt /tmp/exp5/local_files/file2.txt /user/lab/exercise1/


2026-10-09 13:24:52 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
$ hdfs dfs -ls /user/lab/exercise1


2026-10-09 13:24:53 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Found 2 items
-rw-r--r--   1 ojas supergroup        146 2026-10-09 13:24 /user/lab/exercise1/file1.txt
-rw-r--r--   1 ojas supergroup        136 2026-10-09 13:24 /user/lab/exercise1/file2.txt


CompletedProcess(args='hdfs dfs -ls /user/lab/exercise1', returncode=0, stdout='2026-10-09 13:24:53 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable\nFound 2 items\n-rw-r--r--   1 ojas supergroup        146 2026-10-09 13:24 /user/lab/exercise1/file1.txt\n-rw-r--r--   1 ojas supergroup        136 2026-10-09 13:24 /user/lab/exercise1/file2.txt\n', stderr='')

### Exercise 2: Copy a file from one HDFS directory to another

In [ ]:
run("hdfs dfs -mkdir -p /user/lab/exercise2")
run("hdfs dfs -cp /user/lab/exercise1/file1.txt /user/lab/exercise2/file1_copied.txt")
run("hdfs dfs -ls /user/lab/exercise2")

$ hdfs dfs -mkdir -p /user/lab/exercise2


2026-10-09 13:24:53 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
$ hdfs dfs -cp /user/lab/exercise1/file1.txt /user/lab/exercise2/file1_copied.txt


2026-10-09 13:24:54 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
$ hdfs dfs -ls /user/lab/exercise2


2026-10-09 13:24:54 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Found 1 items
-rw-r--r--   1 ojas supergroup        146 2026-10-09 13:24 /user/lab/exercise2/file1_copied.txt


CompletedProcess(args='hdfs dfs -ls /user/lab/exercise2', returncode=0, stdout='2026-10-09 13:24:54 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable\nFound 1 items\n-rw-r--r--   1 ojas supergroup        146 2026-10-09 13:24 /user/lab/exercise2/file1_copied.txt\n', stderr='')

### Exercise 3: Delete a directory from HDFS

In [ ]:
run("hdfs dfs -mkdir -p /user/lab/to_delete")
run(f"hdfs dfs -put -f {quote(str(LOCAL_DIR / 'sample.txt'))} /user/lab/to_delete/")
print("Before delete:")
run("hdfs dfs -ls /user/lab")
run("hdfs dfs -rm -r /user/lab/to_delete")
print("\nAfter delete:")
run("hdfs dfs -ls /user/lab")

$ hdfs dfs -mkdir -p /user/lab/to_delete


2026-10-09 13:24:55 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
$ hdfs dfs -put -f /tmp/exp5/local_files/sample.txt /user/lab/to_delete/


2026-10-09 13:24:55 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Before delete:
$ hdfs dfs -ls /user/lab


2026-10-09 13:24:56 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Found 5 items
drwxr-xr-x   - ojas supergroup          0 2026-10-09 13:24 /user/lab/exercise1
drwxr-xr-x   - ojas supergroup          0 2026-10-09 13:24 /user/lab/exercise2
drwxr-xr-x   - ojas supergroup          0 2026-10-09 13:24 /user/lab/input
drwxr-xr-x   - ojas supergroup          0 2026-10-09 13:24 /user/lab/output
drwxr-xr-x   - ojas supergroup          0 2026-10-09 13:24 /user/lab/to_delete
$ hdfs dfs -rm -r /user/lab/to_delete


2026-10-09 13:24:57 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Deleted /user/lab/to_delete

After delete:
$ hdfs dfs -ls /user/lab


2026-10-09 13:24:57 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Found 4 items
drwxr-xr-x   - ojas supergroup          0 2026-10-09 13:24 /user/lab/exercise1
drwxr-xr-x   - ojas supergroup          0 2026-10-09 13:24 /user/lab/exercise2
drwxr-xr-x   - ojas supergroup          0 2026-10-09 13:24 /user/lab/input
drwxr-xr-x   - ojas supergroup          0 2026-10-09 13:24 /user/lab/output


CompletedProcess(args='hdfs dfs -ls /user/lab', returncode=0, stdout='2026-10-09 13:24:57 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable\nFound 4 items\ndrwxr-xr-x   - ojas supergroup          0 2026-10-09 13:24 /user/lab/exercise1\ndrwxr-xr-x   - ojas supergroup          0 2026-10-09 13:24 /user/lab/exercise2\ndrwxr-xr-x   - ojas supergroup          0 2026-10-09 13:24 /user/lab/input\ndrwxr-xr-x   - ojas supergroup          0 2026-10-09 13:24 /user/lab/output\n', stderr='')

### Exercise 4: Display the contents of an HDFS file

In [ ]:
run("hdfs dfs -cat /user/lab/exercise1/file2.txt")

$ hdfs dfs -cat /user/lab/exercise1/file2.txt


2026-10-09 13:24:58 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
File 2: hadoop notes
HDFS = Hadoop Distributed File System
YARN = Yet Another Resource Negotiator
MapReduce = parallel processing model


CompletedProcess(args='hdfs dfs -cat /user/lab/exercise1/file2.txt', returncode=0, stdout='2026-10-09 13:24:58 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable\nFile 2: hadoop notes\nHDFS = Hadoop Distributed File System\nYARN = Yet Another Resource Negotiator\nMapReduce = parallel processing model\n', stderr='')

### Exercise 5: Check the available space in HDFS

In [ ]:
run("hdfs dfs -df -h /")
print()
run("hdfs dfsadmin -report")

$ hdfs dfs -df -h /


2026-10-09 13:24:58 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Filesystem                Size   Used  Available  Use%
hdfs://localhost:9000  460.4 G  5.0 K     87.4 G    0%

$ hdfs dfsadmin -report


2026-10-09 13:24:59 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Configured Capacity: 494384795648 (460.43 GB)
Present Capacity: 93866075119 (87.42 GB)
DFS Remaining: 93866070016 (87.42 GB)
DFS Used: 5103 (4.98 KB)
DFS Used%: 0.00%
Replicated Blocks:
	Under replicated blocks: 0
	Blocks with corrupt replicas: 0
	Missing blocks: 0
	Missing blocks (with replication factor 1): 0
	Low redundancy blocks with highest priority to recover: 0
	Pending deletion blocks: 0
Erasure Coded Block Groups: 
	Low redundancy block groups: 0
	Block groups with corrupt internal blocks: 0
	Missing block groups: 0
	Low redundancy blocks with highest priority to recover: 0
	Pending deletion blocks: 0

-------------------------------------------------
Live datanodes (1):

Name: 127.0.0.1:9866 (localhost)
Hostname: localhost
Decommission Status : Normal
Configured Capacity: 494384795648 (460.43 GB)
DFS Used: 5103 (4.98 KB)
Non DF

CompletedProcess(args='hdfs dfsadmin -report', returncode=0, stdout='2026-10-09 13:24:59 WARN  NativeCodeLoader:60 - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable\nConfigured Capacity: 494384795648 (460.43 GB)\nPresent Capacity: 93866075119 (87.42 GB)\nDFS Remaining: 93866070016 (87.42 GB)\nDFS Used: 5103 (4.98 KB)\nDFS Used%: 0.00%\nReplicated Blocks:\n\tUnder replicated blocks: 0\n\tBlocks with corrupt replicas: 0\n\tMissing blocks: 0\n\tMissing blocks (with replication factor 1): 0\n\tLow redundancy blocks with highest priority to recover: 0\n\tPending deletion blocks: 0\nErasure Coded Block Groups: \n\tLow redundancy block groups: 0\n\tBlock groups with corrupt internal blocks: 0\n\tMissing block groups: 0\n\tLow redundancy blocks with highest priority to recover: 0\n\tPending deletion blocks: 0\n\n-------------------------------------------------\nLive datanodes (1):\n\nName: 127.0.0.1:9866 (localhost)\nHostname: localhost\nD

---
## Observation

- Java JDK and Hadoop were installed and configured for a single-node setup.
- Hadoop XML files (`core-site.xml`, `hdfs-site.xml`) were configured.
- The NameNode was formatted and HDFS daemons (NameNode, DataNode) started successfully.
- Directories were created in HDFS.
- Files were uploaded, listed, read, copied, downloaded, and deleted using HDFS shell commands.
- All exercise operations completed without errors.

## Result

Hadoop was successfully configured, and all basic HDFS operations such as creating directories, uploading, listing, reading, downloading, and deleting files were performed successfully.

## Conclusion

The experiment demonstrated the installation and configuration of Hadoop in a single-node environment. Basic HDFS operations were successfully performed using Hadoop shell commands, providing practical understanding of distributed storage and file management in Hadoop.